# CS224N · Lecture 12 — Efficient Neural Network Training

**Slides:** [cs224n-spr2024-lecture12-training-shikhar.pdf](https://web.stanford.edu/class/cs224n/slides/cs224n-spr2024-lecture12-training-shikhar.pdf) (Shikhar Murty; the PEFT part adapted from slides by Diyi Yang)  
*"Hopefully useful for final projects!"*

The three techniques that let you train and adapt models far larger than your hardware would naively allow:

| § | Topic | You will implement / verify (no GPU needed) |
|---|---|---|
| 1 | **Floating point 101**: FP32, FP16, BF16 | Bit-exact FP16 and BF16 rounding; range and precision, measured |
| 2 | **Mixed-precision training**: master weights, loss scaling, BF16 | Train the same network in 5 precision regimes; dynamic loss scaling |
| 3 | **Multi-GPU training**: DDP, ZeRO stages 1–3 / FSDP | Ring all-reduce; DDP, ZeRO-1, and FSDP simulated across "GPUs", verified **identical** to single-device training; memory and communication accounting |
| 4 | **Parameter-efficient finetuning: LoRA** | LoRA from scratch; accuracy vs. trainable parameters; the "low intrinsic rank" of updates; merging for zero extra latency |
| 5 | The decision flowchart | — |
| 6 | PyTorch recipes | AMP + GradScaler, BF16 autocast, DDP/FSDP launch skeleton, a LoRA layer |

Everything is simulated in NumPy so you can see *exactly* what each technique does to the numbers; §6 shows the few lines of PyTorch you'd actually write.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math, time, collections

np.set_printoptions(precision=6, suppress=False)
rng = np.random.default_rng(0)

---
## 1. Floating points 101

A floating-point number is stored as **sign × 2^exponent × (1.mantissa)**:

| Format | Sign | Exponent bits | Mantissa bits | Bytes | |
|---|---|---|---|---|---|
| **FP32** | 1 | 8 | 23 | 4 | the default |
| **FP16** (half) | 1 | 5 | 10 | 2 | |
| **BF16** (bfloat16) | 1 | **8** | 7 | 2 | same range as FP32, less precision |

* The **exponent** bits determine the **range** (how large and how small numbers can get).
* The **mantissa** bits determine the **precision**: FP32 can represent $[2^E, 2^E(1+\epsilon), 2^E(1+2\epsilon), \dots, 2^{E+1}]$ with $\epsilon = 2^{-23}$. Between consecutive powers of two there are $2^{23}$ evenly spaced numbers, and nothing in between.

NumPy has FP16 (`np.float16`) but not BF16, so we emulate BF16 exactly: a BF16 number is the **top 16 bits** of an FP32 number, with round-to-nearest-even.

In [ ]:
def to_fp16(x):
    return np.asarray(x, np.float32).astype(np.float16).astype(np.float32)

def to_bf16(x):
    # Round FP32 -> BF16 (keep the top 16 bits, round-to-nearest-even), returned as float32 for convenience.
    b = np.asarray(x, np.float32).copy().view(np.uint32)
    rounding = ((b >> 16) & 1) + np.uint32(0x7FFF)
    return ((b + rounding) & np.uint32(0xFFFF0000)).view(np.float32)

def bits(x, fmt):
    if fmt == "fp32":
        u = int(np.float32(x).view(np.uint32)); s = f"{u:032b}"; return f"{s[0]} {s[1:9]} {s[9:]}"
    if fmt == "fp16":
        u = int(np.float16(x).view(np.uint16)); s = f"{u:016b}"; return f"{s[0]} {s[1:6]} {s[6:]}"
    if fmt == "bf16":
        u = int(to_bf16(x).view(np.uint32)) >> 16; s = f"{u:016b}"; return f"{s[0]} {s[1:9]} {s[9:]}"

for x in [1.0, 0.1, -3.140625]:
    print(f"{x:>10}:  fp32 {bits(x, 'fp32')}\n{'':12s}fp16 {bits(x, 'fp16')}\n{'':12s}bf16 {bits(x, 'bf16')}")

In [ ]:
formats = {
    "fp32": dict(eps=2.0 ** -23, max=float(np.finfo(np.float32).max), min_normal=float(np.finfo(np.float32).tiny), min_sub=2.0 ** -149),
    "fp16": dict(eps=2.0 ** -10, max=float(np.finfo(np.float16).max), min_normal=float(np.finfo(np.float16).tiny), min_sub=2.0 ** -24),
    "bf16": dict(eps=2.0 ** -7, max=float(to_bf16(np.float32(3.389e38))), min_normal=float(np.finfo(np.float32).tiny), min_sub=2.0 ** -133),
}
print(f"{'format':6s} {'epsilon (precision)':>20s} {'largest':>12s} {'smallest normal':>16s} {'smallest subnormal':>19s}")
for k, f in formats.items():
    print(f"{k:6s} {f['eps']:20.3e} {f['max']:12.3e} {f['min_normal']:16.3e} {f['min_sub']:19.3e}")

print("\nThe slide's examples:")
print(f"  1.0001 in fp16 = {to_fp16(1.0001):.6f}    in bf16 = {to_bf16(1.0001):.6f}    in fp32 = {np.float32(1.0001):.6f}")
print(f"  a tiny gradient 1e-8: fp16 -> {to_fp16(1e-8):.3e} (underflow!)   bf16 -> {to_bf16(1e-8):.3e}")
print(f"  a big activation 70000: fp16 -> {to_fp16(70000.0)} (overflow!)   bf16 -> {to_bf16(70000.0):.0f}")
w, update = np.float32(1.0), np.float32(1e-4)
print(f"  weight update 1.0 + 1e-4: fp32 -> {w + update:.6f}   fp16 -> {to_fp16(to_fp16(w) + to_fp16(update)):.6f}   bf16 -> {to_bf16(to_bf16(w) + to_bf16(update)):.6f}")

**Training neural networks in half precision?** Standard training represents parameters and gradients in FP32, which causes CUDA out-of-memory errors with large models. **Possible solution: use FP16!** But:
* **Less range:** FP16 covers roughly $6\times10^{-8}$ (subnormal) / $6\times10^{-5}$ (normal) up to $6.5\times10^4$. Above that is `inf`.
* **Smaller precision → rounding errors:** 1.0001 is 1 in half precision.
* For training: **gradients can underflow** (become exactly 0), and **weight updates are imprecise** (a small update added to a large weight vanishes).

BF16 has FP32's range (no underflow/overflow worries in practice) but even *less* precision than FP16 (ε = 2⁻⁷ ≈ 0.008), so it has the weight-update problem too.

---
## 2. Mixed-precision training (Micikevicius et al. 2018)

**Still use FP16, but use FP32 for neural network updates!** The recipe:
1. Maintain a copy of model parameters in FP32 (**master weights**).
2. Run the forward pass in FP16.
3. **Scale the loss by a large value** (to artificially increase the gradient).
4. Compute the gradient in FP16.
5. Copy the gradient into FP32 and **divide by the scale factor**.
6. Update the master weights in FP32 [fixes the weight-update issue!].
7. Copy into the FP16 version.

Without step 3, **gradients can still underflow** (small gradients become exactly 0). **Can we get rid of gradient scaling?** We need scaling because FP16 has a small range compared to FP32. **Can we allocate 8 bits for the exponent (same range) while sacrificing precision?** That's **BF16**: it can represent much smaller and much larger numbers (no INF/NaNs), so **BF16 does not need gradient scalers**.

### 2.1 The experiment
To see every failure mode, we train a 2-layer MLP regressor with Adam in five regimes. We simulate "computing in format X" by rounding every forward/backward tensor to X (standard "fake quantization"; real hardware accumulates matrix products in FP32 internally, which we also mimic by rounding after each op, not inside it). The network has only 4 hidden units, so its hidden layer *must* learn, and the task's targets are small (scale $10^{-2}$), so its gradients are small, as many real gradients are (Micikevicius et al. showed large fractions of activation gradients in real networks fall below FP16's range).

| Regime | Compute (forward/backward) | Weights updated in | Loss scaling |
|---|---|---|---|
| FP32 baseline | FP32 | FP32 | — |
| pure FP16 | FP16 | **FP16** | none |
| FP16 + FP32 master weights | FP16 | FP32 | **none** |
| FP16 mixed precision | FP16 | FP32 | **×2¹⁶, dynamic** |
| BF16 mixed precision | BF16 | FP32 | none needed |

In [ ]:
D_IN, H = 16, 4                                       # few hidden units: the hidden layer MUST learn
r = np.random.default_rng(0)
W_teacher = r.normal(size=(D_IN, 1)) / np.sqrt(D_IN)
X_all = r.normal(size=(4096, D_IN)).astype(np.float32)
y_all = (1e-2 * np.tanh(X_all @ W_teacher)).astype(np.float32)          # small-scale targets -> small gradients

def init_mlp(seed=0):
    r = np.random.default_rng(seed)
    return {"W1": r.normal(0, 1 / np.sqrt(D_IN), (D_IN, H)).astype(np.float32), "b1": np.zeros(H, np.float32),
            "W2": r.normal(0, 1e-3 / np.sqrt(H), (H, 1)).astype(np.float32), "b2": np.zeros(1, np.float32)}

def mlp_loss_grads(p, X, y, q=lambda t: t, loss_scale=1.0):
    # q() rounds a tensor to the compute format. Returns (true loss, grads of loss*loss_scale in compute format).
    P = {k: q(v) for k, v in p.items()}                                   # cast weights to compute format
    Xq = q(X)
    z = q(Xq @ P["W1"] + P["b1"]); h = q(np.maximum(z, 0)); out = q(h @ P["W2"] + P["b2"])
    diff = out - y
    loss = float(np.mean(diff.astype(np.float64) ** 2))
    d_out = q(loss_scale * 2 * diff / len(y))                             # gradient of the SCALED loss
    g = {"W2": q(h.T @ d_out), "b2": q(d_out.sum(0))}
    dh = q(d_out @ P["W2"].T); dz = q(dh * (z > 0))
    g["W1"] = q(Xq.T @ dz); g["b1"] = q(dz.sum(0))
    return loss, g

class AdamState:
    def __init__(self, p):
        self.m = {k: np.zeros_like(v) for k, v in p.items()}; self.v = {k: np.zeros_like(v) for k, v in p.items()}; self.t = 0

def adam_update(p, g, st, lr=1e-3, weight_q=lambda t: t):
    st.t += 1
    for k in p:
        st.m[k] = 0.9 * st.m[k] + 0.1 * g[k]; st.v[k] = 0.999 * st.v[k] + 0.001 * g[k] ** 2
        step = lr * (st.m[k] / (1 - 0.9 ** st.t)) / (np.sqrt(st.v[k] / (1 - 0.999 ** st.t)) + 1e-12)
        p[k] = weight_q(p[k] - step)                                       # weight_q = format the weights are stored in

def train(regime, steps=1500, batch=256):
    p = init_mlp(); st = AdamState(p); r = np.random.default_rng(1)
    q = {"fp32": lambda t: t, "fp16": to_fp16, "bf16": to_bf16}[regime["compute"]]
    wq = {"fp32": lambda t: t, "fp16": to_fp16}[regime["weights"]]
    scale, good_steps, scales, losses, zero_frac = regime.get("scale", 1.0), 0, [], [], []
    p = {k: wq(v) for k, v in p.items()}
    for step in range(steps):
        idx = r.integers(0, len(X_all), batch)
        loss, g = mlp_loss_grads(p, X_all[idx], y_all[idx], q=q, loss_scale=scale)
        zero_frac.append(np.mean(g["W1"] == 0))
        finite = all(np.isfinite(v).all() for v in g.values())
        if regime.get("dynamic") and not finite:                 # overflow: skip this step, halve the scale
            scale /= 2; good_steps = 0; scales.append(scale); losses.append(loss); continue
        g = {k: v.astype(np.float32) / scale for k, v in g.items()}       # unscale in FP32
        adam_update(p, g, st, weight_q=wq)
        if regime.get("dynamic"):
            good_steps += 1
            if good_steps % 100 == 0:
                scale *= 2                                       # periodically try a larger scale
        scales.append(scale); losses.append(loss)
    return np.array(losses), np.array(scales), np.mean(zero_frac)

regimes = {
    "FP32 baseline": dict(compute="fp32", weights="fp32"),
    "pure FP16": dict(compute="fp16", weights="fp16"),
    "FP16 + FP32 master, no scaling": dict(compute="fp16", weights="fp32"),
    "FP16 mixed precision (dynamic scaling)": dict(compute="fp16", weights="fp32", scale=2.0 ** 16, dynamic=True),
    "BF16 mixed precision, no scaling": dict(compute="bf16", weights="fp32"),
}
results = {}
t0 = time.time()
for name, reg in regimes.items():
    results[name] = train(reg)
    l, s, z = results[name]
    print(f"{name:40s} final loss {np.mean(l[-100:]):.3e}   fraction of hidden-layer (W1) gradients exactly 0: {z:.1%}")
print(f"(initial loss {results['FP32 baseline'][0][0]:.3e}; {time.time() - t0:.0f}s)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name, (l, s, z) in results.items():
    ax[0].semilogy(np.convolve(l, np.ones(20) / 20, mode="valid"), label=name)
ax[0].set_xlabel("step"); ax[0].set_ylabel("training loss (smoothed)"); ax[0].legend(fontsize=7); ax[0].set_title("Same model, five precision regimes")
ax[1].semilogy(results["FP16 mixed precision (dynamic scaling)"][1]); ax[1].set_xlabel("step"); ax[1].set_ylabel("loss scale")
ax[1].set_title("Dynamic loss scale (halve on overflow, double every 100 good steps)"); plt.tight_layout(); plt.show()

Reading the results:
* **Without loss scaling** (pure FP16, and FP16 with FP32 master weights), about a third of the hidden layer's gradients are **exactly zero**. They underflowed in the FP16 backward pass, so those weights never learn, and the final loss is several times worse than FP32.
* **Loss scaling** lifts the gradients into FP16's range before the backward pass, and the result matches the FP32 baseline. The dynamic scaler starts at $2^{16}$, halves whenever a gradient overflows to `inf` (skipping that step), and periodically tries doubling again. That's what PyTorch's `GradScaler` does.
* **BF16** needs no scaling at all: its 8-bit exponent has FP32's range.

**Why FP32 master weights?** In this short run, Adam's updates (lr = $10^{-3}$) happen to be large enough to survive FP16 rounding, so pure FP16 fails only from underflow. The weight-update problem appears when many *small* updates must accumulate, as in long training with a decaying learning rate. Here it is in isolation:

In [ ]:
w16, w32 = np.float16(1.0), np.float32(1.0)
for _ in range(1000):                                   # 1000 small updates of +1e-4 each
    w16 = np.float16(w16 + np.float16(1e-4))
    w32 = np.float32(w32 + np.float32(1e-4))
print(f"after 1000 updates of 1e-4:  weight stored in fp16 = {float(w16):.4f}   in fp32 = {float(w32):.4f}   (exact: 1.1000)")
print(f"(in fp16, the spacing between representable numbers near 1.0 is {2 ** -10:.1e}, so +1e-4 rounds back to 1.0 every time)")

PyTorch's AMP recipe (from the docs): wrap the forward pass in `torch.autocast(device_type="cuda", dtype=torch.float16)`, then `scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()` (§6). With BF16, drop the scaler. Real-world result from the slides (finetuning DistilBERT on one A100): mixed precision and BF16 dramatically cut training time and memory with no loss in accuracy.

---
## 3. Multi-GPU training

### 3.1 What's stored on GPU memory?
For a model with $\Psi$ parameters, trained with mixed precision and Adam:

| Item | Bytes per parameter |
|---|---|
| Model parameters (FP16) | 2 |
| Gradients (FP16) | 2 |
| Master weights (FP32) | 4 |
| Adam momentum (FP32) | 4 |
| Adam variance (FP32) | 4 |
| **Total** | **16** |

**Plus model activations, which scale with the batch size!** (Activations are often the largest item; **activation checkpointing**, 🔎, recomputes them during the backward pass instead of storing them.)

In [ ]:
def training_memory_gb(n_params, bytes_per_param=16):
    return n_params * bytes_per_param / 1e9

for name, n in [("BERT-large (340M)", 340e6), ("Llama 7B", 7e9), ("Llama 70B", 70e9)]:
    print(f"{name:18s}: weights alone (fp16) {n * 2 / 1e9:6.1f} GB   full training state {training_memory_gb(n):7.1f} GB   (+ activations)")
print("A single A100/H100 has 40-80 GB.")

### 3.2 Distributed Data Parallel (DDP)
**Each GPU has a synchronized copy of the model, with its own slice of the data.**
1. **Forward pass in parallel** on each GPU's slice.
2. **Backward pass**, while communicating gradients for upstream parameters as soon as they're ready (overlapping communication with computation).
3. An **all-reduce** operation sums the gradients across GPUs, so every GPU ends up with the gradient of the full batch. **Communication overhead: 2 bytes per parameter** (gradients in FP16).
4. Each GPU's optimizer applies the same update, **maintaining synchronization**.

**How does all-reduce work?** The standard **ring all-reduce** = a **reduce-scatter** (each GPU ends up with the *sum* of one $1/N$ chunk) followed by an **all-gather** (every GPU collects all the summed chunks). Each GPU sends $2\frac{N-1}{N}$ times the data size in total, **independent of the number of GPUs**, which is why it scales. Let's implement it and verify:

In [ ]:
class Comm:
    # Counts the bytes each simulated GPU sends.
    def __init__(self, n):
        self.n, self.sent = n, np.zeros(n)

def reduce_scatter(comm, bufs):
    # Ring reduce-scatter: after N-1 steps, GPU i holds the full SUM of chunk (i+1) % N.
    n = comm.n; chunks = [np.array_split(b.copy(), n) for b in bufs]
    for step in range(n - 1):
        msgs = [(i, (i - step) % n, chunks[i][(i - step) % n].copy()) for i in range(n)]   # send one chunk to the right
        for i, c, data in msgs:
            chunks[(i + 1) % n][c] += data; comm.sent[i] += data.nbytes
    return [chunks[i][(i + 1) % n] for i in range(n)]            # GPU i owns chunk (i+1) % n

def all_gather(comm, owned, sizes):
    # Ring all-gather: GPU i starts with chunk (i+1) % N; after N-1 steps every GPU has every chunk.
    n = comm.n; have = [{(i + 1) % n: owned[i]} for i in range(n)]
    for step in range(n - 1):
        msgs = [(i, (i + 1 - step) % n, have[i][(i + 1 - step) % n]) for i in range(n)]
        for i, c, data in msgs:
            have[(i + 1) % n][c] = data; comm.sent[i] += data.nbytes
    return [np.concatenate([have[i][c] for c in range(n)]) for i in range(n)]

def all_reduce(comm, bufs):
    sizes = [len(c) for c in np.array_split(bufs[0], comm.n)]
    return all_gather(comm, reduce_scatter(comm, bufs), sizes)

for n in [2, 4, 8]:
    comm = Comm(n)
    bufs = [np.random.default_rng(i).normal(size=10_000).astype(np.float32) for i in range(n)]
    out = all_reduce(comm, bufs)
    ok = all(np.allclose(o, np.sum(bufs, 0), atol=1e-4) for o in out)
    print(f"{n} GPUs: all-reduce correct: {ok};  bytes sent per GPU / buffer size = {comm.sent.mean() / bufs[0].nbytes:.3f}"
          f"   (theory 2(N-1)/N = {2 * (n - 1) / n:.3f})")

**DDP, simulated.** Four "GPUs", each with a full replica of an MLP and its own Adam state, each processing one quarter of every batch. After the all-reduce, every replica has the full-batch gradient. **The replicas stay identical, and match single-GPU training on the full batch:**

In [ ]:
# A small classifier MLP with parameters as one flat vector (that's how DDP/FSDP buckets and shards them).
SHAPES = {"W1": (20, 64), "b1": (64,), "W2": (64, 5), "b2": (5,)}
SIZES = {k: int(np.prod(s)) for k, s in SHAPES.items()}
N_PARAMS = sum(SIZES.values())
def unflatten(v):
    out, i = {}, 0
    for k, s in SHAPES.items():
        out[k] = v[i:i + SIZES[k]].reshape(s); i += SIZES[k]
    return out
def flatten(d):
    return np.concatenate([d[k].ravel() for k in SHAPES])

r = np.random.default_rng(0)
Xc = r.normal(size=(4096, 20)); Wt = r.normal(size=(20, 5)); yc = np.argmax(np.tanh(Xc @ Wt) + 0.3 * r.normal(size=(4096, 5)), 1)

def loss_grad_flat(theta, X, y):
    p = unflatten(theta)
    z = X @ p["W1"] + p["b1"]; h = np.maximum(z, 0); logits = h @ p["W2"] + p["b2"]
    P = np.exp(logits - logits.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
    loss = -np.log(P[np.arange(len(y)), y]).mean()
    G = P; G[np.arange(len(y)), y] -= 1; G /= len(y)
    dh = (G @ p["W2"].T) * (z > 0)
    return loss, flatten({"W1": X.T @ dh, "b1": dh.sum(0), "W2": h.T @ G, "b2": G.sum(0)})

def adam_flat(theta, g, m, v, t, lr=3e-3):
    m = 0.9 * m + 0.1 * g; v = 0.999 * v + 0.001 * g ** 2
    return theta - lr * (m / (1 - 0.9 ** t)) / (np.sqrt(v / (1 - 0.999 ** t)) + 1e-8), m, v

theta0 = np.random.default_rng(1).normal(0, 0.1, N_PARAMS)
batches = [np.random.default_rng(100 + s).integers(0, len(Xc), 256) for s in range(200)]

# single "GPU", full batch
theta, m, v = theta0.copy(), np.zeros(N_PARAMS), np.zeros(N_PARAMS)
for t, idx in enumerate(batches, 1):
    _, g = loss_grad_flat(theta, Xc[idx], yc[idx]); theta, m, v = adam_flat(theta, g, m, v, t)
theta_single = theta

# DDP across N simulated GPUs
N_GPU = 4
comm = Comm(N_GPU)
replicas = [theta0.copy() for _ in range(N_GPU)]; ms = [np.zeros(N_PARAMS) for _ in range(N_GPU)]; vs = [np.zeros(N_PARAMS) for _ in range(N_GPU)]
for t, idx in enumerate(batches, 1):
    shards = np.array_split(idx, N_GPU)                                          # each GPU gets its slice of the batch
    local_grads = [loss_grad_flat(replicas[i], Xc[s], yc[s])[1] for i, s in enumerate(shards)]
    summed = all_reduce(comm, local_grads)                                        # every GPU gets the SUM
    for i in range(N_GPU):
        replicas[i], ms[i], vs[i] = adam_flat(replicas[i], summed[i] / N_GPU, ms[i], vs[i], t)
print(f"DDP: replicas identical: {all(np.allclose(replicas[0], rp) for rp in replicas)};  "
      f"max |DDP - single GPU| = {np.abs(replicas[0] - theta_single).max():.2e}")
print(f"DDP communication per step per GPU: {comm.sent.mean() / len(batches) / 1e3:.1f} KB "
      f"(= 2(N-1)/N x {N_PARAMS} params x 8 bytes)")

### 3.3 Unfortunately, naive DDP has poor memory scaling
Every GPU holds all 16 bytes/parameter. The **ZeRO** optimizer (Rajbhandari et al. 2020, DeepSpeed; also implemented as PyTorch FSDP) removes this redundancy in three stages:

**ZeRO stage 1: optimizer-state sharding ($P_{os}$).**
* Each GPU has the full set of FP16 parameters and computes the gradient on its subset of the data.
* Each GPU has a **sharded copy of the optimizer state** (master weights + Adam moments), and is responsible for updating one shard of the parameters.
* Each worker computes the gradient on its data; a **reduce-scatter** gives each worker the full (summed) gradient for *its* parameter shard; each worker updates its shard; an **all-gather** synchronizes the parameters.
* Communication overhead: still **2 bytes per parameter** (reduce-scatter + all-gather moves the same data as one all-reduce): **we saved memory for free!**

**ZeRO stage 2: optimizer-state + gradient sharding ($P_{os+g}$).** Complexity: we still need the full gradient for the worker's data slice! Solution: **never instantiate the full gradient vector**. During the backward pass, layer by layer: compute the gradient for layer $j$'s parameters, **immediately send it to the GPU in charge** of that shard (reduce), and **deallocate** it. Then each worker updates its shard, and an all-gather synchronizes.

**ZeRO stage 3 (full FSDP): when even the model parameters won't fit.** Caveat: so far communication overhead was "free"; with full FSDP, it no longer is. High-level sketch:
1. Divide model parameters into **FSDP units**; 2. **shard** each unit across GPUs;
3. **Forward pass:** **all-gather** so each GPU gets all pieces of a unit, run it, **discard** the gathered shards;
4. **Backward pass:** all-gather again, compute gradients on each GPU's data chunk, **reduce-scatter** to send each gradient piece to the right GPU;
5. Each GPU updates its own shard using the full gradient received.

**Communication recap:** DDP: all-reduce. ZeRO-1/2: reduce-scatter + all-gather (memory saved for free). ZeRO-3: all-gather + reduce-scatter + all-gather (more overhead!).

Let's simulate **ZeRO-1** and **ZeRO-3/FSDP** and verify both produce exactly the same model as DDP:

In [ ]:
def zero1(batches, n):
    comm = Comm(n)
    params = [theta0.copy() for _ in range(n)]                                        # full params on every GPU
    shard_sizes = [len(c) for c in np.array_split(theta0, n)]
    own = [(i + 1) % n for i in range(n)]                                             # chunk index each GPU owns (ring layout)
    starts = np.cumsum([0] + shard_sizes)
    m = [np.zeros(shard_sizes[own[i]]) for i in range(n)]; v = [np.zeros(shard_sizes[own[i]]) for i in range(n)]
    for t, idx in enumerate(batches, 1):
        shards = np.array_split(idx, n)
        local_grads = [loss_grad_flat(params[i], Xc[s], yc[s])[1] for i, s in enumerate(shards)]
        grad_shards = reduce_scatter(comm, local_grads)                                # each GPU: summed grad of ITS shard
        new_shards = []
        for i in range(n):
            c = own[i]; sl = slice(starts[c], starts[c + 1])
            upd, m[i], v[i] = adam_flat(params[i][sl], grad_shards[i] / n, m[i], v[i], t)   # optimizer state only for the shard
            new_shards.append(upd)
        params = all_gather(comm, new_shards, shard_sizes)                             # re-synchronize full params
    return params, comm

def fsdp(batches, n):
    comm = Comm(n)
    shard_sizes = [len(c) for c in np.array_split(theta0, n)]
    own = [(i + 1) % n for i in range(n)]; starts = np.cumsum([0] + shard_sizes)
    param_shard = [theta0[starts[own[i]]:starts[own[i] + 1]].copy() for i in range(n)]  # each GPU stores ONLY its shard
    m = [np.zeros_like(s) for s in param_shard]; v = [np.zeros_like(s) for s in param_shard]
    peak_resident = 0
    for t, idx in enumerate(batches, 1):
        full = all_gather(comm, param_shard, shard_sizes)                              # gather params just in time
        peak_resident = max(peak_resident, len(full[0]))
        shards = np.array_split(idx, n)
        local_grads = [loss_grad_flat(full[i], Xc[s], yc[s])[1] for i, s in enumerate(shards)]
        del full                                                                       # discard gathered params
        grad_shards = reduce_scatter(comm, local_grads)
        for i in range(n):
            param_shard[i], m[i], v[i] = adam_flat(param_shard[i], grad_shards[i] / n, m[i], v[i], t)
    final = all_gather(comm, param_shard, shard_sizes)
    return final, comm

p_z1, comm_z1 = zero1(batches, N_GPU)
p_fsdp, comm_fsdp = fsdp(batches, N_GPU)
print(f"ZeRO-1: max |ZeRO-1 - single GPU| = {np.abs(p_z1[0] - theta_single).max():.2e}   comm per step per GPU {comm_z1.sent.mean() / len(batches) / 1e3:.1f} KB")
print(f"FSDP  : max |FSDP   - single GPU| = {np.abs(p_fsdp[0] - theta_single).max():.2e}   comm per step per GPU {comm_fsdp.sent.mean() / len(batches) / 1e3:.1f} KB")
print(f"DDP   : comm per step per GPU {comm.sent.mean() / len(batches) / 1e3:.1f} KB")

Same final model, to floating-point precision, in all three schemes: sharding changes **where** state lives and **what** is communicated, not the math.

Notice that ZeRO-1 communicates exactly as much as DDP (a reduce-scatter plus an all-gather moves the same data as one all-reduce): **the memory savings are free**. In our simulation FSDP also sends the same amount, because it gathers the full parameters once per step and keeps them for both the forward and the backward pass. A real FSDP **frees** each unit's gathered parameters right after using them in the forward pass (that's the point: never hold the whole model), so it must **all-gather them again for the backward pass**. That's the slide's "all-gather + reduce-scatter + all-gather": about **1.5× DDP's communication**. FSDP trades communication for memory, and overlaps that communication with computation to hide it.

### 3.4 Memory per GPU, for real model sizes
With $K = 12$ bytes/param of optimizer state (FP32 master weights + 2 Adam moments) and $N$ GPUs (the ZeRO paper's accounting):

| Scheme | Bytes per parameter per GPU |
|---|---|
| DDP | $2 + 2 + K = 16$ |
| ZeRO-1 ($P_{os}$) | $2 + 2 + K/N$ |
| ZeRO-2 ($P_{os+g}$) | $2 + (2 + K)/N$ |
| ZeRO-3 / FSDP ($P_{os+g+p}$) | $(2 + 2 + K)/N$ |

In [ ]:
def per_gpu_gb(n_params, n_gpus, stage, K=12):
    b = {"DDP": 2 + 2 + K, "ZeRO-1": 2 + 2 + K / n_gpus, "ZeRO-2": 2 + (2 + K) / n_gpus, "ZeRO-3/FSDP": (2 + 2 + K) / n_gpus}[stage]
    return n_params * b / 1e9

for model, n_params in [("Llama 7B", 7e9), ("Llama 70B", 70e9)]:
    for n_gpus in [8, 64]:
        row = "  ".join(f"{st} {per_gpu_gb(n_params, n_gpus, st):7.1f}" for st in ["DDP", "ZeRO-1", "ZeRO-2", "ZeRO-3/FSDP"])
        print(f"{model:10s} on {n_gpus:2d} GPUs (GB per GPU, before activations): {row}")

A 7B model doesn't fit on one 80GB GPU for full training with DDP (112 GB), fits comfortably with ZeRO-2 on 8 GPUs, and a 70B model needs ZeRO-3 across many GPUs. Remember **activations** come on top, scaling with batch size and sequence length (use activation checkpointing, smaller micro-batches with **gradient accumulation**, or both).

🔎 Beyond data parallelism, very large training runs also split each layer's matrices across GPUs (**tensor parallelism**, Megatron-LM) and assign different layers to different GPUs (**pipeline parallelism**). Large runs combine all three ("3D parallelism").

### 3.5 🔎 Beyond the slides: four more tools you'll use in every real training run
**(a) Learning-rate warmup + cosine decay.** Almost every Transformer is trained with a schedule: **linear warmup** from 0 to the peak learning rate over the first ~1–5% of steps, then **cosine decay** to ~10% of peak. Warmup matters because Adam's second-moment estimates are unreliable in the first steps (their bias-corrected values come from very few gradients), so full-size steps then can destabilize training, especially for post-LN Transformers (L8).

In [ ]:
def lr_schedule(step, total, peak=3e-4, warmup_frac=0.02, final_frac=0.1):
    warm = max(1, int(total * warmup_frac))
    if step < warm:
        return peak * (step + 1) / warm                                    # linear warmup
    progress = (step - warm) / max(1, total - warm)
    return peak * (final_frac + (1 - final_frac) * 0.5 * (1 + math.cos(math.pi * progress)))   # cosine decay

total = 10_000
plt.figure(figsize=(6, 3)); plt.plot([lr_schedule(t, total) for t in range(total)])
plt.xlabel("step"); plt.ylabel("learning rate"); plt.title("Linear warmup (2%) + cosine decay to 10%"); plt.show()

**(b) Gradient accumulation.** If the batch you want doesn't fit in memory, split it into micro-batches, **sum** their gradients (each scaled by its share of the batch), and step once. For per-example-averaged losses this is *exactly* the large-batch gradient. Caveats: BatchNorm statistics differ (LayerNorm is per-example, so Transformers are fine), and with per-token losses over variable-length sequences you must weight micro-batches by their **token counts**, not equally.

In [ ]:
idx = batches[0]                                                       # a batch of 256 from the DDP example
_, g_full = loss_grad_flat(theta0, Xc[idx], yc[idx])
g_acc = np.zeros_like(g_full)
for micro in np.array_split(idx, 4):                                    # 4 micro-batches of 64, processed one at a time
    _, g_m = loss_grad_flat(theta0, Xc[micro], yc[micro])
    g_acc += g_m * len(micro) / len(idx)                                # weight by the micro-batch's share
print(f"gradient accumulation (4 x 64) vs one batch of 256: max difference {np.abs(g_acc - g_full).max():.1e}")

**(c) Activation checkpointing** (Chen et al. 2016). Storing every layer's activations for the backward pass costs memory linear in depth $L$. Instead, store activations only at every $k$-th layer, and **recompute** the rest during the backward pass. With $k\approx\sqrt{L}$, activation memory drops from $O(L)$ to $O(\sqrt{L})$, for roughly one extra forward pass (~30% more compute). In PyTorch: `torch.utils.checkpoint.checkpoint(block, x)`.

**(d) Measuring efficiency: MFU.** **Model FLOPs utilization** = the training FLOPs the model *needs* per second ($\approx 6N$ × tokens/second) divided by the hardware's peak FLOPs. Good large-scale runs reach roughly 35–55%; a low MFU means the GPUs are waiting on communication, memory, or the data loader.

In [ ]:
def activation_memory(L, k):
    # units = activations of one layer; checkpoint every k layers, recompute one segment at a time
    return math.ceil(L / k) + k
L_layers = 64
print(f"{L_layers} layers: store all = {L_layers} layer-activations; checkpoint every 8 = {activation_memory(L_layers, 8)}; "
      f"every 4 = {activation_memory(L_layers, 4)}  (optimum near sqrt(L) = {int(math.sqrt(L_layers))})")

def mfu(n_params, tokens_per_sec_per_gpu, peak_flops):
    return 6 * n_params * tokens_per_sec_per_gpu / peak_flops
print(f"MFU of a 7B model at 3,000 tokens/s per A100 (312 TFLOP/s bf16 peak): {mfu(7e9, 3000, 312e12):.0%}")
print(f"tokens/s per GPU needed for 50% MFU on an H100 (~989 TFLOP/s dense bf16): {0.5 * 989e12 / (6 * 7e9):,.0f}")

### 3.6 Multi-GPU training optimizations: recap
* **Always use mixed-precision training.** Always use BF16 if `torch.cuda.is_bf16_supported()`.
* Does batch size 1 fit on a single GPU? **Yes** → try a larger batch size / use ZeRO stage 2. **No** → does ZeRO stage 3 solve the OOM errors? **No** → try **parameter-efficient finetuning!**

---
## 4. Parameter-efficient finetuning (PEFT)

**Full finetuning** updates all model parameters. **Parameter-efficient finetuning** updates a small subset. Why finetune only some parameters?
1. **Fine-tuning all parameters is impractical with large models.**
2. **State-of-the-art models are massively overparameterized** → parameter-efficient finetuning **matches the performance** of full finetuning.

**Why do we need efficient adaptation?** The maximum training compute for the largest AI models grew ~2× every 3.4 months, vs. global compute capacity ~2× every 1.5 years: a clearly unsustainable rate, forecast to slow. As training costs rise, AI development concentrates in the most well-funded organizations (whose value systems are then embedded in the AI of tomorrow?). Current AI papers emphasize accuracy over efficiency, and there are hidden environmental costs (one 2021 estimate put GPT-3's training emissions on par with running a coal power plant for 10 hours). The slides' example: in a Stanford RL class of 200+ students, two algorithms performed equally well, but one used far more power; if everyone had used the efficient one, they'd have saved **880 kWh**, about a typical American household's monthly use.

### 4.1 Full finetuning, formally
Assume a pretrained autoregressive LM $P_\Phi(y\mid x)$ (e.g., GPT), adapted to a downstream task (summarization, NL2SQL, reading comprehension) with a dataset of context–target pairs $\{(x_i, y_i)\}_{i=1..N}$. During **full finetuning**, we update $\Phi_0$ to $\Phi_0 + \Delta\Phi$ by following the gradient to maximize the conditional LM objective:
$$\max_\Phi\sum_{(x,y)}\sum_{t=1}^{|y|}\log P_\Phi(y_t\mid x, y_{<t})$$
For **each** downstream task we learn a different $\Delta\Phi$ with $|\Delta\Phi| = |\Phi_0|$. GPT-3 has $|\Phi_0|$ = 175 billion: expensive and challenging for storing and deploying many independent instances. **Can we do better?**

**Key idea:** encode the task-specific increment $\Delta\Phi = \Delta\Phi(\Theta)$ with a much smaller set of parameters $\Theta$, $|\Theta|\ll|\Phi_0|$, and optimize over $\Theta$:
$$\max_\Theta\sum_{(x,y)}\sum_{t=1}^{|y|}\log P_{\Phi_0 + \Delta\Phi(\Theta)}(y_t\mid x, y_{<t})$$

### 4.2 LoRA: low-rank-parameterized update matrices (Hu et al. 2021)
Updates to the weights have a **low "intrinsic rank"** during adaptation (Aghajanyan et al. 2020). For a pretrained weight matrix $W_0\in\mathbb{R}^{d\times k}$, constrain its update with a low-rank decomposition:
$$W_0 + \Delta W = W_0 + \alpha BA,\qquad B\in\mathbb{R}^{d\times r},\ A\in\mathbb{R}^{r\times k},\ r\ll\min(d, k)$$
* Only $A$ and $B$ contain trainable parameters: $r(d+k)$ instead of $dk$.
* $\alpha$ trades off pretrained "knowledge" against task-specific "knowledge" (in practice the update is often scaled by $\alpha/r$).
* $A$ is initialized randomly and **$B$ at zero**, so training starts exactly at the pretrained model.
* As one increases the number of trainable parameters (the rank), training LoRA **converges to training the original model**.
* **No additional inference latency:** merge $W = W_0 + \alpha BA$ once. When switching to a different task, recover $W_0$ by subtracting $BA$ and adding a different $B'A'$.
* Often LoRA is applied to the weight matrices in the **self-attention** module.

### 4.3 LoRA from scratch
We "pretrain" a 2-layer MLP on task A, then adapt it to a related task B with few examples. Task B's true function differs from A's by a **low-rank change** to the first layer (the situation LoRA assumes). We compare: training only the output head (linear probe), LoRA on the first layer at ranks 1–8 (plus the head), and full finetuning.

In [ ]:
D_IN2, H2, C2 = 32, 128, 4
r = np.random.default_rng(0)
W1_teacher = r.normal(size=(D_IN2, H2)) / np.sqrt(D_IN2)
U_, V_ = r.normal(size=(D_IN2, 2)), r.normal(size=(2, H2))
W1_teacher_B = W1_teacher + 0.6 * U_ @ V_ / np.sqrt(D_IN2)                 # task B = task A + a rank-2 change
W2_teacher = r.normal(size=(H2, C2)) / np.sqrt(H2)
def make_task(W1t, n, seed):
    r_ = np.random.default_rng(seed); X = r_.normal(size=(n, D_IN2))
    return X, np.argmax(np.maximum(X @ W1t, 0) @ W2_teacher + 0.1 * r_.normal(size=(n, C2)), 1)
XA, yA = make_task(W1_teacher, 20000, 1)
XB_tr, yB_tr = make_task(W1_teacher_B, 400, 2); XB_te, yB_te = make_task(W1_teacher_B, 5000, 3)

def forward(p, X):
    W1 = p["W1"] + (p["B"] @ p["A"]) * p["scale"] if "A" in p else p["W1"]
    z = X @ W1 + p["b1"]; h = np.maximum(z, 0)
    return z, h, h @ p["W2"] + p["b2"], W1

def grads(p, X, y):
    z, h, logits, W1 = forward(p, X)
    P = np.exp(logits - logits.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
    G = P.copy(); G[np.arange(len(y)), y] -= 1; G /= len(y)
    dz = (G @ p["W2"].T) * (z > 0)
    g = {"W2": h.T @ G, "b2": G.sum(0), "b1": dz.sum(0)}
    dW1 = X.T @ dz
    g["W1"] = dW1
    if "A" in p:                                      # chain rule through W1 = W0 + scale * B A
        g["B"] = p["scale"] * dW1 @ p["A"].T
        g["A"] = p["scale"] * p["B"].T @ dW1
    return g

def accuracy(p, X, y):
    return np.mean(forward(p, X)[2].argmax(1) == y)

def adam_train(p, X, y, keys, steps, lr, seed=0, batch=128):
    r_ = np.random.default_rng(seed); m = {k: np.zeros_like(p[k]) for k in keys}; v = {k: np.zeros_like(p[k]) for k in keys}
    for t in range(1, steps + 1):
        idx = r_.integers(0, len(y), batch); g = grads(p, X[idx], y[idx])
        for k in keys:
            m[k] = 0.9 * m[k] + 0.1 * g[k]; v[k] = 0.999 * v[k] + 0.001 * g[k] ** 2
            p[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
    return p

# "pretraining" on task A
r = np.random.default_rng(5)
p_pre = {"W1": r.normal(0, 1 / np.sqrt(D_IN2), (D_IN2, H2)), "b1": np.zeros(H2), "W2": r.normal(0, 1 / np.sqrt(H2), (H2, C2)), "b2": np.zeros(C2)}
p_pre = adam_train(p_pre, XA, yA, ["W1", "b1", "W2", "b2"], 3000, 3e-3)
print(f"pretrained model: task A accuracy {accuracy(p_pre, *make_task(W1_teacher, 5000, 9)):.1%};  "
      f"task B accuracy before adaptation {accuracy(p_pre, XB_te, yB_te):.1%}")

In [ ]:
def copy(p):
    return {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in p.items()}

rows = []
p = adam_train(copy(p_pre), XB_tr, yB_tr, ["W2", "b2"], 800, 3e-3)
rows.append(("head only (linear probe)", (H2 + 1) * C2, accuracy(p, XB_te, yB_te)))
for rank in [1, 2, 4, 8]:
    p = copy(p_pre); r_ = np.random.default_rng(rank)
    p["A"] = r_.normal(0, 1 / np.sqrt(H2), (rank, H2)); p["B"] = np.zeros((D_IN2, rank)); p["scale"] = 1.0
    out0 = forward(p, XB_te)[2]
    assert np.allclose(out0, forward(p_pre, XB_te)[2]), "B=0 init must reproduce the pretrained model exactly"
    p = adam_train(p, XB_tr, yB_tr, ["A", "B", "W2", "b2"], 800, 3e-3)
    rows.append((f"LoRA rank {rank} on W1 (+ head)", rank * (D_IN2 + H2) + (H2 + 1) * C2, accuracy(p, XB_te, yB_te)))
    if rank == 2:
        p_lora2 = p
p_full = adam_train(copy(p_pre), XB_tr, yB_tr, ["W1", "b1", "W2", "b2"], 800, 3e-3)
rows.append(("full finetuning", D_IN2 * H2 + H2 + (H2 + 1) * C2, accuracy(p_full, XB_te, yB_te)))
print(f"{'method':30s} {'trainable params':>17s} {'task B test accuracy':>21s}   (400 training examples)")
for name, n_p, acc in rows:
    print(f"{name:30s} {n_p:17,d} {acc:21.1%}")

### 4.4 Understanding low-rank adaptation
Two checks of LoRA's assumptions and claims:
1. **Does full finetuning's update really have low intrinsic rank?** Look at the singular values of $\Delta W_1 = W_1^{\text{finetuned}} - W_1^{\text{pretrained}}$.
2. **No extra inference latency:** merge $BA$ into $W_1$ and check that the outputs are unchanged.

In [ ]:
dW = p_full["W1"] - p_pre["W1"]
s = np.linalg.svd(dW, compute_uv=False)
energy = np.cumsum(s ** 2) / np.sum(s ** 2)
print("singular values of the full-finetuning update dW1 (top 8):", s[:8].round(3))
dW_noise = adam_train(copy(p_pre), XA[:400], yA[:400], ["W1", "b1", "W2", "b2"], 800, 3e-3)["W1"] - p_pre["W1"]
s_noise = np.linalg.svd(dW_noise, compute_uv=False)
print("same procedure on 400 examples of the ORIGINAL task A (no real change needed), top 8:", s_noise[:8].round(3))
print(f"fraction of the update's energy in the top 1 / 2 / 4 / 8 directions: "
      f"{energy[0]:.1%} / {energy[1]:.1%} / {energy[3]:.1%} / {energy[7]:.1%}   (of {min(dW.shape)} possible)")

merged = copy(p_lora2); merged["W1"] = p_lora2["W1"] + p_lora2["scale"] * p_lora2["B"] @ p_lora2["A"]
for k in ["A", "B", "scale"]:
    merged.pop(k)
diff = np.abs(forward(merged, XB_te)[2] - forward(p_lora2, XB_te)[2]).max()
print(f"merged LoRA model vs. unmerged: max output difference {diff:.1e}  -> deploy as a plain matrix, zero extra latency")
unmerged = copy(merged); unmerged["W1"] = merged["W1"] - p_lora2["scale"] * p_lora2["B"] @ p_lora2["A"]
print(f"subtracting BA recovers the pretrained W1 exactly: {np.allclose(unmerged['W1'], p_pre['W1'])}")
plt.figure(figsize=(6, 3)); plt.semilogy(s, "o-", ms=3); plt.xlabel("index"); plt.ylabel("singular value")
plt.title("Spectrum of the full-finetuning update"); plt.show()

**Reading the spectrum honestly.** The full-finetuning update is *not* cleanly low-rank: its top directions stand out a bit, but most of its energy is spread across many directions. The control row shows why. Finetuning on 400 examples of the *original* task, where no change is needed at all, also produces a sizable update spread across many directions. That part is fitting noise in a small dataset, not task signal. So full finetuning = (a low-rank useful change) + (high-rank noise). LoRA restricts the update to $r$ directions, which acts as a **regularizer**, one reason LoRA at rank 1–4 matched or beat full finetuning here with a fraction of the trainable parameters. (Aghajanyan et al. measured low intrinsic dimension for large *pretrained* language models, where the effect is much stronger than in this toy MLP.)

**LoRA in practice** (Hu et al. 2021): on GPT-2 medium/large (E2E NLG Challenge), LoRA outperforms several baselines (adapters, prefix tuning) with comparable or fewer trainable parameters. On GPT-3 175B, LoRA **matches or exceeds the full-finetuning baseline** on WikiSQL, MultiNLI, and SAMSum, while training ~10,000× fewer parameters, and it **scales better** than prefix-based methods as the number of trainable parameters grows.

🔎 **The PEFT family and what came after:**
* **Adapters** (Houlsby et al. 2019): small bottleneck MLPs inserted into each layer (adds inference latency, unlike merged LoRA).
* **Prefix tuning / prompt tuning** (Li & Liang 2021; Lester et al. 2021): learn continuous "virtual tokens" prepended to the input or to every layer's keys and values.
* **QLoRA** (Dettmers et al. 2023): keep the frozen base model in **4-bit** (NF4) precision and train LoRA adapters in BF16, which made finetuning a 65B model possible on a single 48GB GPU.
* **DoRA** (Liu et al. 2024): decompose weights into magnitude and direction, applying LoRA to the direction.

---
## 5. Summarizing everything

* **Always use mixed-precision training.** Always use BF16 if `torch.cuda.is_bf16_supported()`.
* **Does batch size = 1 fit on a single GPU?**
    * **Yes:** try a larger batch size / use ZeRO stage 2.
    * **No:** does ZeRO stage 3 solve the OOM errors? **No** → **try LoRA!**
* **Even simpler:** *start with Llama 7B + bfloat16 + ZeRO stage 3 (or FSDP) + LoRA* 🫡

| Technique | What it saves | Cost |
|---|---|---|
| Mixed precision (FP16 + loss scaling, or BF16) | ~half the activation/param memory, much faster matmuls | FP32 master weights; loss scaling for FP16 |
| DDP | wall-clock time (more data per step) | 2 bytes/param communication; full memory per GPU |
| ZeRO-1/2 | optimizer state / gradient memory, ÷N | same communication as DDP: free |
| ZeRO-3 / FSDP | parameter memory too, ÷N | extra all-gathers every step |
| LoRA | trainable params, optimizer state, checkpoint size per task | small accuracy risk if the rank is too low |

---
## 6. PyTorch recipes

### 6.1 Automatic mixed precision (AMP)
The PyTorch AMP recipe from the slides. `autocast` runs eligible ops (matmuls, convolutions) in low precision and keeps numerically sensitive ones (softmax, layer norm, losses) in FP32. `GradScaler` implements dynamic loss scaling (§2). With BF16, skip the scaler.

In [ ]:
# [PyTorch] AMP training loop: FP16 + GradScaler on CUDA, BF16 if supported (falls back to FP32 on CPU).
import torch, torch.nn as nn, torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 5)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda" and not use_bf16))   # only FP16 needs scaling
Xt, yt = torch.tensor(Xc, dtype=torch.float32, device=device), torch.tensor(yc, device=device)

for step in range(200):
    idx = torch.randint(0, len(Xt), (256,), device=device)
    with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
        loss = F.cross_entropy(model(Xt[idx]), yt[idx])            # forward in FP16/BF16 (weights stay FP32)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()                                  # scale the loss, backward in low precision
    scaler.step(opt)                                               # unscale; skip the step if grads are inf/NaN
    scaler.update()                                                # adjust the scale factor
print(f"device={device}, dtype={'bf16' if use_bf16 else ('fp16' if device == 'cuda' else 'fp32 (no CUDA)')}, final loss {loss.item():.4f}")

### 6.2 DDP and FSDP: the launch skeleton
These need multiple processes, so they're shown as a script rather than run in the notebook. Save as `train.py` and launch with `torchrun --nproc_per_node=4 train.py`. Each process drives one GPU; `DistributedSampler` gives each its slice of the data.

```python
import os, torch, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP, MixedPrecision, ShardingStrategy

dist.init_process_group("nccl")
rank = int(os.environ["LOCAL_RANK"]); torch.cuda.set_device(rank)
model = build_model().cuda()

if USE_FSDP:   # ZeRO-3 style: shard params, grads, optimizer state
    model = FSDP(model, sharding_strategy=ShardingStrategy.FULL_SHARD,     # SHARD_GRAD_OP ~ ZeRO-2
                 mixed_precision=MixedPrecision(param_dtype=torch.bfloat16, reduce_dtype=torch.bfloat16))
else:          # DDP: full replica per GPU, all-reduce of gradients during backward
    model = DDP(model, device_ids=[rank])

opt = torch.optim.AdamW(model.parameters(), lr=1e-4)        # create the optimizer AFTER wrapping
sampler = torch.utils.data.distributed.DistributedSampler(dataset)
loader = torch.utils.data.DataLoader(dataset, batch_size=8, sampler=sampler)
for epoch in range(num_epochs):
    sampler.set_epoch(epoch)                                 # different shuffle each epoch
    for batch in loader:
        with torch.autocast("cuda", dtype=torch.bfloat16):
            loss = model(**batch).loss
        loss.backward(); opt.step(); opt.zero_grad()
dist.destroy_process_group()
```

### 6.3 A LoRA layer in PyTorch

In [ ]:
# [PyTorch] LoRA wrapper for nn.Linear, with merge/unmerge.
import math, torch, torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)                                # freeze the pretrained weights
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))   # B = 0: start exactly at the pretrained model
        self.scale = alpha / r
        self.merged = False
    def forward(self, x):
        out = self.base(x)
        return out if self.merged else out + (x @ self.A.T @ self.B.T) * self.scale
    @torch.no_grad()
    def merge(self):                                               # fold BA into W: zero extra inference latency
        if not self.merged:
            self.base.weight += (self.B @ self.A) * self.scale; self.merged = True

torch.manual_seed(0)
lin = nn.Linear(64, 32); lora = LoRALinear(lin, r=4)
x = torch.randn(5, 64)
print("initial output equals the frozen base layer:", torch.allclose(lora(x), lin(x)))
with torch.no_grad():
    lora.B.normal_()                                               # pretend we trained B
y_unmerged = lora(x); lora.merge(); y_merged = lora(x)
n_train = sum(p.numel() for p in lora.parameters() if p.requires_grad)
print(f"merged == unmerged: {torch.allclose(y_unmerged, y_merged, atol=1e-5)};  trainable params {n_train} vs {lin.weight.numel()} in W")
# With Hugging Face PEFT:  from peft import LoraConfig, get_peft_model
#   model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"]))

---
## 7. Summary

| Concept | Key point |
|---|---|
| FP32 / FP16 / BF16 | Exponent bits = range, mantissa bits = precision; FP16: 5/10, BF16: 8/7 |
| Half-precision failure modes | Gradient underflow (→ 0), imprecise updates (1.0001 = 1), overflow (→ inf) |
| Mixed precision | FP32 master weights + low-precision compute + loss scaling (FP16) or none (BF16) |
| Training memory | 16 bytes/param (2+2+4+4+4) + activations |
| DDP | Replicas + all-reduce; ring all-reduce sends $2(N-1)/N$ × size per GPU |
| ZeRO-1/2 | Shard optimizer state (and gradients): same communication, much less memory |
| ZeRO-3 / FSDP | Shard parameters too; all-gather just in time: more communication |
| LoRA | $W_0 + \alpha BA$, $B = 0$ init; few trainable params; merge for zero latency |

### Exercises
1. In §2, make the targets 100× smaller. Find the smallest static loss scale that recovers FP32-level training in FP16.
2. Add an **overflow** scenario to §2: multiply the inputs by 1000. Which regimes produce `inf`, and does dynamic loss scaling recover?
3. Implement **ZeRO-2** in §3: shard the gradients so no GPU ever holds the full gradient vector (send each layer's gradient to its owner as soon as it's computed).
4. Implement **gradient accumulation**: reproduce a batch of 256 with 4 micro-batches of 64 on one GPU, and verify the update matches.
5. In §4, make task B differ from task A by a **full-rank** change. How does the LoRA-vs-full-finetuning comparison change, and what does the spectrum of $\Delta W$ look like?
6. Quantize the pretrained $W_1$ to 4 bits (16 uniform levels per column) and train LoRA on top (a mini QLoRA). How much accuracy do you lose?

### References
* Micikevicius et al. 2018, *Mixed Precision Training*; Kalamkar et al. 2019 (BF16 for deep learning).
* Rajbhandari et al. 2020, *ZeRO: Memory Optimizations Toward Training Trillion Parameter Models*; Zhao et al. 2023 (PyTorch FSDP).
* Shoeybi et al. 2019 (Megatron-LM, tensor parallelism); Huang et al. 2019 (GPipe, pipeline parallelism).
* Hu et al. 2021, *LoRA*; Aghajanyan et al. 2020 (intrinsic dimensionality); Dettmers et al. 2023 (QLoRA).
* Houlsby et al. 2019 (adapters); Li & Liang 2021 (prefix tuning); Lester et al. 2021 (prompt tuning).
* Henderson et al. 2020, *Towards the Systematic Reporting of the Energy and Carbon Footprints of Machine Learning*.
* PyTorch AMP docs (`torch.amp`) and FSDP tutorial; Raschka 2023, *Accelerating LLMs with Mixed-Precision Techniques* (blog).

**Next:** Lecture 13 onward (speech and brain–computer interfaces, agents, life after DPO, ...).